# ⚡ Parallel Fan-Out and Fan-In

## Learning Objectives
In this notebook, you will learn:
1. **Fan-out / fan-in** - Send one input to several independent agents in the same super-step, then join their results once
2. **Reducers** - Why parallel nodes that write the same state field need `Annotated[list, operator.add]`, proved without an LLM
3. **Map-reduce with `Send`** - Fan out to a number of workers known only at run time, one per document
4. **When to choose it** - Independent work, a clear merge rule, and what parallelism does and does not save

## Where this fits
This is pattern 3 of [Five Practical Agent Patterns](../../../../docs/source/08/five-practical-agent-patterns.md). Notebooks 01 and 02 ran one agent at a time. Here several prebuilt `create_agent` agents work **at the same time** on the same input, and a final node merges what they found.

**Flow:** `Input → workers A, B, C concurrently → join / reducer → result`

---
## 📦 Step 1: Environment Setup

The setup cell imports everything the notebook uses, loads the API keys from the project-root `.env` and sets `RECURSION_LIMIT`, the step cap every agent run passes.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and imports
# ============================================================================
import operator
import time
import warnings
from typing import Annotated, TypedDict

from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
from langgraph.errors import InvalidUpdateError
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Every agent run passes this cap on LangGraph super-steps
RECURSION_LIMIT = 25

print(f"✅ Environment ready (recursion_limit = {RECURSION_LIMIT})")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: one chat model shared by every agent
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

In [ ]:
# ============================================================================
# ASK: run a prebuilt agent on one message and return its final reply
# ============================================================================
def ask(agent, text: str) -> str:
    """Send one user message to a create_agent agent; return the text of its last message."""
    result = agent.invoke({"messages": [HumanMessage(content=text)]}, {"recursion_limit": RECURSION_LIMIT})
    return result["messages"][-1].content


print("✅ ask() ready")

---
## 🌿 Part 1: Fan-Out to Three Analysts, Fan-In to One Lead

A product team reviews an idea from three angles. The three analyses do not depend on each other, so they can run at the same time:

```
            ┌→ market_analyst ─┐
START ──────┼→ tech_analyst   ─┼──→ product_lead → END
            └→ risk_analyst   ─┘
```

- **Fan-out:** three edges leave `START`. LangGraph runs every node whose inputs are ready in the same **super-step**, and it runs the nodes of one super-step concurrently.
- **Fan-in:** `add_edge([...three analysts...], "product_lead")` is a **join**. `product_lead` waits until all three have finished, then runs once.

Each analyst node also records `config["metadata"]["langgraph_step"]`, the super-step it ran in. If the three ran in parallel, they share one step number.

In [ ]:
# ============================================================================
# ANALYST AGENTS: three independent views, plus a lead who merges them
# ============================================================================
ANALYST_PROMPTS = {
    "market_analyst": "You are a market analyst. Assess customer demand and competition for the idea.",
    "tech_analyst": "You are a senior engineer. Assess how hard the idea is to build and run.",
    "risk_analyst": "You are a risk analyst. Name the biggest legal, privacy and business risks.",
}
analysts = {
    name: create_agent(llm, system_prompt=prompt + " Reply in at most 3 short bullet points.", name=name)
    for name, prompt in ANALYST_PROMPTS.items()
}
product_lead = create_agent(
    llm,
    system_prompt=(
        "You are the product lead. Read the analyses and give a GO or NO-GO verdict, "
        "then explain it in at most 3 sentences."
    ),
    name="product_lead",
)

print(f"✅ Agents ready: {list(analysts)} + product_lead")

In [ ]:
# ============================================================================
# FAN-OUT / FAN-IN GRAPH
# ============================================================================
class ReviewState(TypedDict):
    idea: str
    analyses: Annotated[list[dict], operator.add]  # reducer: parallel writes are concatenated (Part 2)
    verdict: str
    lead_step: int


def make_analyst_node(name: str):
    agent = analysts[name]

    def analyst(state: ReviewState, config: RunnableConfig) -> dict:
        text = ask(agent, f"Product idea: {state['idea']}")
        step = config["metadata"]["langgraph_step"]  # the super-step this node ran in
        return {"analyses": [{"analyst": name, "step": step, "text": text}]}

    return analyst


def lead(state: ReviewState, config: RunnableConfig) -> dict:
    joined = "\n\n".join(f"{a['analyst']}:\n{a['text']}" for a in state["analyses"])
    verdict = ask(product_lead, f"Idea: {state['idea']}\n\nAnalyses:\n{joined}")
    return {"verdict": verdict, "lead_step": config["metadata"]["langgraph_step"]}


builder = StateGraph(ReviewState)
for name in analysts:
    builder.add_node(name, make_analyst_node(name))
    builder.add_edge(START, name)  # fan-out: three edges from START
builder.add_node("product_lead", lead)
builder.add_edge(list(analysts), "product_lead")  # fan-in: wait for all three, then run once
builder.add_edge("product_lead", END)
review_app = builder.compile()

print(review_app.get_graph().draw_mermaid())

In [ ]:
# ============================================================================
# RUN: ONE IDEA, THREE PARALLEL ANALYSTS, ONE VERDICT
# ============================================================================
IDEA = "A mobile app that uses the phone camera to estimate the calories in a meal photo."

start = time.perf_counter()
result = review_app.invoke({"idea": IDEA, "analyses": []})
elapsed = time.perf_counter() - start

for analysis in result["analyses"]:
    print(f"🔎 {analysis['analyst']} (super-step {analysis['step']}):\n{analysis['text']}\n")
print(f"🧑‍💼 product_lead (super-step {result['lead_step']}):\n{result['verdict']}")
print(f"\n⏱️ whole run: {elapsed:.1f}s")

steps = {a["step"] for a in result["analyses"]}
print(f"\n{'✅' if len(result['analyses']) == 3 else '❌'} three analyses were merged")
print(f"{'✅' if len(steps) == 1 else '❌'} all analysts ran in the same super-step: {steps}")
print(f"{'✅' if result['lead_step'] == max(steps) + 1 else '❌'} the lead ran once, in the next super-step")

The three analysts share one super-step, so their three model calls overlapped. The whole run takes roughly the time of the **slowest** analyst plus the lead, not the sum of all four. The cost does not shrink: each analyst still makes its own model call.

---
## ➕ Part 2: Why the Reducer Matters

All three analysts write to `analyses` in the same super-step. LangGraph needs a rule for merging those writes; the **reducer** in `Annotated[list[dict], operator.add]` is that rule: concatenate the lists.

Without a reducer, a field accepts one value per super-step. Two parallel writes to it are a conflict, and LangGraph raises `InvalidUpdateError` rather than silently keeping one of them. The two tiny graphs below prove this with plain functions and no LLM.

In [ ]:
# ============================================================================
# REDUCER PROOF: the same two parallel writers, without and with a reducer (no LLM)
# ============================================================================
class NoReducer(TypedDict):
    results: list[str]  # no reducer: one value per super-step


class WithReducer(TypedDict):
    results: Annotated[list[str], operator.add]  # reducer: concatenate parallel writes


def build_two_writers(state_schema):
    builder = StateGraph(state_schema)
    builder.add_node("worker_a", lambda state: {"results": ["a"]})
    builder.add_node("worker_b", lambda state: {"results": ["b"]})
    for worker in ["worker_a", "worker_b"]:
        builder.add_edge(START, worker)
        builder.add_edge(worker, END)
    return builder.compile()


try:
    build_two_writers(NoReducer).invoke({"results": []})
    print("❌ no error without a reducer")
except InvalidUpdateError as error:
    print(f"✅ without a reducer: InvalidUpdateError: {str(error).splitlines()[0]}")

merged = build_two_writers(WithReducer).invoke({"results": []})["results"]
print(f"{'✅' if sorted(merged) == ['a', 'b'] else '❌'} with operator.add: {merged}")

---
## 🗺️ Part 3: Map-Reduce With `Send`

Part 1 had exactly three workers, wired with three edges. Often the number of workers depends on the input: one summary per customer review, one search per sub-question. `Send` handles that.

- A **fan-out function** returns a list of `Send("summarize", payload)` objects, one per review. Each `Send` starts its own `summarize` task with its own small payload, not the whole graph state.
- `add_conditional_edges(START, fan_out, ["summarize"])` attaches that function to `START`.
- The `summarize` tasks run in parallel and append to `summaries` through the reducer. `report` runs once after all of them.

In [ ]:
# ============================================================================
# MAP-REDUCE GRAPH: one summarize task per review, then one report
# ============================================================================
summarizer = create_agent(
    llm,
    system_prompt=(
        "Summarize the customer review in one sentence. "
        "Start with 'Positive:', 'Negative:' or 'Mixed:'."
    ),
    name="summarizer",
)
reporter = create_agent(
    llm,
    system_prompt=(
        "You write a product feedback report from one-line review summaries. "
        "Give the count of positive, negative and mixed reviews, then the top 2 themes. At most 5 lines."
    ),
    name="reporter",
)


class MapReduceState(TypedDict):
    reviews: list[str]
    summaries: Annotated[list[str], operator.add]  # each summarize task appends one line
    report: str


class ReviewTask(TypedDict):
    review: str  # the only thing one summarize task receives


def fan_out(state: MapReduceState) -> list[Send]:
    return [Send("summarize", {"review": review}) for review in state["reviews"]]


def summarize(task: ReviewTask) -> dict:
    return {"summaries": [ask(summarizer, task["review"])]}


def report(state: MapReduceState) -> dict:
    return {"report": ask(reporter, "\n".join(state["summaries"]))}


builder = StateGraph(MapReduceState)
builder.add_node("summarize", summarize)
builder.add_node("report", report)
builder.add_conditional_edges(START, fan_out, ["summarize"])
builder.add_edge("summarize", "report")
builder.add_edge("report", END)
map_reduce_app = builder.compile()

print("✅ Map-reduce graph compiled")

In [ ]:
# ============================================================================
# RUN: FIVE REVIEWS, THEN TWO (the worker count follows the input)
# ============================================================================
REVIEWS = [
    "Battery lasts two full days. Best purchase this year.",
    "Stopped charging after a week. Support never replied.",
    "Great screen, but the speaker is too quiet for calls.",
    "Fast delivery and the setup took two minutes.",
    "The case cracked on the first drop. Disappointed.",
]

for reviews in [REVIEWS, REVIEWS[:2]]:
    result = map_reduce_app.invoke({"reviews": reviews, "summaries": []})
    print("=" * 80)
    for line in result["summaries"]:
        print(f"• {line}")
    print(f"\n📊 report:\n{result['report']}")
    ok = len(result["summaries"]) == len(reviews)
    print(f"\n{'✅' if ok else '❌'} {len(reviews)} reviews → {len(result['summaries'])} summarize tasks")

The graph did not change between the two runs; only the input did. Five reviews started five `summarize` tasks and two reviews started two. That is the difference between static fan-out (Part 1, one edge per worker) and map-reduce with `Send` (one task per item).

---
## 🧑‍⚖️ Part 4: When to Choose Parallel Fan-Out

| Question | Fan-out fits when... |
|---|---|
| Do workers need each other's results? | No. If B needs A's output, use a sequence (notebook 01). |
| Is the merge rule clear? | Yes: concatenate, vote, or let one lead agent combine. |
| Is the worker count fixed? | Fixed → edges from one node (Part 1). Depends on input → `Send` (Part 3). |
| What does it save? | Wall-clock time. It does not save model calls: every worker still pays for its own. |

---
## 📝 Summary

In this notebook, we learned:

### 1. Fan-out / fan-in
- Several edges from one node start nodes in the same super-step, and LangGraph runs them concurrently. The `langgraph_step` metadata proved the three analysts shared a step.
- `add_edge([a, b, c], "product_lead")` is a join: the lead runs once, after all three.

### 2. Reducers
- Parallel writes to one field need a reducer such as `operator.add`; without one, LangGraph raises `InvalidUpdateError`.

### 3. Map-reduce with `Send`
- A fan-out function returns one `Send(node, payload)` per item, so the number of workers follows the input.

### 4. When to choose it
- Use fan-out for independent work with a clear merge rule. It saves time, not model calls.

### Next Steps
- **`04_generator_critic_loop.ipynb`**: add a loop: one agent writes, another reviews, and the draft is revised until it passes